In [4]:
import subprocess
import sys
import os
import gc
import re
import json
import time
import random
import platform
import shutil
import zipfile
import numpy as np
import pandas as pd
import torch

import transformers
import accelerate
import peft
import bitsandbytes

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, StoppingCriteria, StoppingCriteriaList
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [5]:
GLOBAL_SEED = 42

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(GLOBAL_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def print_gpu_status(label=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"[{label}] Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB | Total: {total:.2f} GB")

def assert_memory_released(label, baseline_gb=0.0, tolerance_gb=0.3):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        delta = allocated - baseline_gb
        if delta > tolerance_gb:
            print(f"WARNING at {label}: {allocated:.2f} GB allocated ({delta:.2f} GB above baseline {baseline_gb:.2f} GB).")
        else:
            print(f"[{label}] Allocated: {allocated:.2f} GB (expected)")

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU name:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total VRAM: {total_vram:.2f} GB")

print("Random seed:", GLOBAL_SEED)
MAX_SAFE_VRAM_GB = 15.9

Python version: 3.10.20
PyTorch version: 2.5.1+cu121
Transformers version: 4.46.3
PEFT version: 0.19.1
CUDA available: True
CUDA version: 12.1
GPU name: Tesla T4
Total VRAM: 14.84 GB
Random seed: 42


In [6]:
TRAIN_PATH = r"C:/Users/VGST/Desktop/AI4Industrial Automation/Training-Dataset-AI4Industrial-Automation-FIRE-2026.csv"
TEST_PATH = r"C:/Users/VGST/Desktop/AI4Industrial Automation/Test-Dataset-AI4Industrial-Automation.csv"

OUTPUT_DIR = r"C:/Users/VGST/Desktop/AI4Industrial Automation/outputs"
CHECKPOINT_DIR = r"C:/Users/VGST/Desktop/AI4Industrial Automation/checkpoints"
PREDICTIONS_CACHE_DIR = os.path.join(OUTPUT_DIR, "predictions_cache")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_CACHE_DIR, exist_ok=True)

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"Training dataset not found at {TRAIN_PATH}")
if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(f"Test dataset not found at {TEST_PATH}")

print("Training path verified:", TRAIN_PATH)
print("Test path verified:", TEST_PATH)

Training path verified: C:/Users/VGST/Desktop/AI4Industrial Automation/Training-Dataset-AI4Industrial-Automation-FIRE-2026.csv
Test path verified: C:/Users/VGST/Desktop/AI4Industrial Automation/Test-Dataset-AI4Industrial-Automation.csv


In [7]:
def robust_read_csv(path):
    for enc in ["utf-8", "utf-8-sig", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc), enc
        except Exception:
            continue
    raise IOError(f"Could not read {path} with any known encoding")

train_df, train_encoding = robust_read_csv(TRAIN_PATH)
test_df, test_encoding = robust_read_csv(TEST_PATH)

print("Training dataset loaded using encoding:", train_encoding)
print("Test dataset loaded using encoding:", test_encoding)
print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

Training dataset loaded using encoding: utf-8
Test dataset loaded using encoding: utf-8
Training shape: (352, 3)
Test shape: (30, 7)


In [8]:
def inspect_dataframe(df, name):
    print("=" * 80)
    print(f"DATASET: {name}")
    print("=" * 80)
    print("Shape:", df.shape)
    print("Column names:", list(df.columns))
    print("\nData types:\n", df.dtypes)
    print("\nMissing values per column:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\nSample records:")
    print(df.head(3).to_string())

inspect_dataframe(train_df, "TRAIN")
inspect_dataframe(test_df, "TEST")

DATASET: TRAIN
Shape: (352, 3)
Column names: ['TASK', 'ST-CODE', 'RUST-CODE']

Data types:
 TASK         object
ST-CODE      object
RUST-CODE    object
dtype: object

Missing values per column:
 TASK         0
ST-CODE      0
RUST-CODE    0
dtype: int64

Duplicate rows: 0

Sample records:
                                                                                          TASK                                                                                                                                                                                                                                                                                                                                                                                                                                                                               ST-CODE                                                                                                                                                   

In [9]:
EXPECTED_TRAIN_COLUMNS = {"TASK", "ST-CODE", "RUST-CODE"}
if not EXPECTED_TRAIN_COLUMNS.issubset(set(train_df.columns)):
    raise ValueError(f"Expected training columns {EXPECTED_TRAIN_COLUMNS} not found. Found: {set(train_df.columns)}")

INPUT_COLUMN_TRAIN = "TASK"
ST_OUTPUT_COLUMN = "ST-CODE"
RUST_OUTPUT_COLUMN = "RUST-CODE"

test_data_columns = [c for c in test_df.columns if not test_df[c].isnull().all()]
test_empty_columns = [c for c in test_df.columns if test_df[c].isnull().all()]

print("Test columns with actual data:", test_data_columns)
print("Test columns that are empty artifacts and will be dropped:", test_empty_columns)

if test_data_columns != ["TASK"]:
    raise ValueError(f"Expected only TASK to carry data in test file. Found data in: {test_data_columns}")

test_df = test_df[["TASK"]].copy()
INPUT_COLUMN_TEST = "TASK"

print("\nFinal confirmed schema:")
print("Train input:", INPUT_COLUMN_TRAIN, "| ST target:", ST_OUTPUT_COLUMN, "| Rust target:", RUST_OUTPUT_COLUMN)
print("Test input:", INPUT_COLUMN_TEST)
print("No ID column in either file. Row index used as identifier.")
print("Every test row requires both a Task A and a Task B prediction.")

Test columns with actual data: ['TASK']
Test columns that are empty artifacts and will be dropped: ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', '#REF!']

Final confirmed schema:
Train input: TASK | ST target: ST-CODE | Rust target: RUST-CODE
Test input: TASK
No ID column in either file. Row index used as identifier.
Every test row requires both a Task A and a Task B prediction.


In [10]:
def clean_text_field(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x == "" or x.lower() in ("nan", "none", "null"):
        return None
    return x

def is_st_code(text):
    if text is None:
        return False
    upper = text.upper()
    return "PROGRAM" in upper and "END_PROGRAM" in upper

def is_rust_code(text):
    if text is None:
        return False
    return ("fn " in text) or ("impl " in text) or ("struct " in text)

train_df["_clean_input"] = train_df[INPUT_COLUMN_TRAIN].apply(clean_text_field)
train_df["_clean_st_output"] = train_df[ST_OUTPUT_COLUMN].apply(clean_text_field)
train_df["_clean_rust_output"] = train_df[RUST_OUTPUT_COLUMN].apply(clean_text_field)

before_rows = len(train_df)
train_df = train_df[train_df["_clean_input"].notnull()].reset_index(drop=True)

invalid_st = (~train_df["_clean_st_output"].apply(is_st_code)).sum()
invalid_rust = (~train_df["_clean_rust_output"].apply(is_rust_code)).sum()
duplicate_input_count = train_df["_clean_input"].duplicated().sum()

print("Rows before cleaning:", before_rows)
print("Rows after removing empty inputs:", len(train_df))
print("Rows where ST-CODE looks malformed:", invalid_st)
print("Rows where RUST-CODE looks malformed:", invalid_rust)
print("Duplicate input descriptions:", duplicate_input_count)

Rows before cleaning: 352
Rows after removing empty inputs: 352
Rows where ST-CODE looks malformed: 0
Rows where RUST-CODE looks malformed: 0
Duplicate input descriptions: 0


In [11]:
task_a_df = train_df[["_clean_input", "_clean_st_output"]].rename(
    columns={"_clean_input": "input", "_clean_st_output": "target"}
).dropna().drop_duplicates().reset_index(drop=True)

task_b_df = train_df[["_clean_input", "_clean_rust_output"]].rename(
    columns={"_clean_input": "input", "_clean_rust_output": "target"}
).dropna().drop_duplicates().reset_index(drop=True)

print("Task A examples (NL to ST):", len(task_a_df))
print("Task B examples (NL to Rust):", len(task_b_df))

if len(task_a_df) == 0 or len(task_b_df) == 0:
    raise ValueError("Task A or Task B construction produced zero examples.")

Task A examples (NL to ST): 352
Task B examples (NL to Rust): 352


In [12]:
def normalize_for_dedup(text):
    return re.sub(r"\s+", " ", text.strip().lower())

all_descriptions = train_df["_clean_input"].apply(normalize_for_dedup).unique()
train_desc, val_desc = train_test_split(all_descriptions, test_size=0.15, random_state=GLOBAL_SEED)
train_desc_set, val_desc_set = set(train_desc), set(val_desc)

task_a_df["_norm_input"] = task_a_df["input"].apply(normalize_for_dedup)
task_b_df["_norm_input"] = task_b_df["input"].apply(normalize_for_dedup)

task_a_train = task_a_df[task_a_df["_norm_input"].isin(train_desc_set)].drop(columns=["_norm_input"]).reset_index(drop=True)
task_a_val = task_a_df[task_a_df["_norm_input"].isin(val_desc_set)].drop(columns=["_norm_input"]).reset_index(drop=True)
task_b_train = task_b_df[task_b_df["_norm_input"].isin(train_desc_set)].drop(columns=["_norm_input"]).reset_index(drop=True)
task_b_val = task_b_df[task_b_df["_norm_input"].isin(val_desc_set)].drop(columns=["_norm_input"]).reset_index(drop=True)

print("Task A train:", len(task_a_train), "val:", len(task_a_val))
print("Task B train:", len(task_b_train), "val:", len(task_b_val))

Task A train: 299 val: 53
Task B train: 299 val: 53


In [13]:
_stat_tokenizer = AutoTokenizer.from_pretrained("gpt2")

def compute_length_stats(df, col):
    if len(df) == 0:
        return {}
    char_lens = df[col].apply(len)
    token_lens = df[col].apply(lambda x: len(_stat_tokenizer.encode(x)))
    return {
        "mean_chars": float(char_lens.mean()), "max_chars": int(char_lens.max()),
        "p95_chars": float(np.percentile(char_lens, 95)),
        "mean_tokens": float(token_lens.mean()), "max_tokens": int(token_lens.max()),
        "p95_tokens": float(np.percentile(token_lens, 95))
    }

for name, df, col in [("Task A input", task_a_df, "input"), ("Task A target", task_a_df, "target"),
                       ("Task B input", task_b_df, "input"), ("Task B target", task_b_df, "target")]:
    print(f"{name} stats:", compute_length_stats(df, col))

def recommend_max_length(df, col, cap=1024):
    if len(df) == 0:
        return 256
    token_lens = df[col].apply(lambda x: len(_stat_tokenizer.encode(x)))
    p95 = int(np.percentile(token_lens, 95))
    return min(max(p95 + 32, 128), cap)

TASK_A_MAX_INPUT_LEN = recommend_max_length(task_a_df, "input", cap=512)
TASK_A_MAX_TARGET_LEN = recommend_max_length(task_a_df, "target", cap=896)
TASK_B_MAX_INPUT_LEN = recommend_max_length(task_b_df, "input", cap=512)
TASK_B_MAX_TARGET_LEN = recommend_max_length(task_b_df, "target", cap=896)

print("TASK_A_MAX_TARGET_LEN:", TASK_A_MAX_TARGET_LEN)
print("TASK_B_MAX_TARGET_LEN:", TASK_B_MAX_TARGET_LEN)

del _stat_tokenizer
clear_gpu_memory()

Token indices sequence length is longer than the specified maximum sequence length for this model (1072 > 1024). Running this sequence through the model will result in indexing errors


Task A input stats: {'mean_chars': 147.14772727272728, 'max_chars': 992, 'p95_chars': 308.69999999999993, 'mean_tokens': 45.88920454545455, 'max_tokens': 235, 'p95_tokens': 90.44999999999999}
Task A target stats: {'mean_chars': 573.2017045454545, 'max_chars': 2262, 'p95_chars': 1070.6499999999992, 'mean_tokens': 258.44602272727275, 'max_tokens': 1072, 'p95_tokens': 497.94999999999965}
Task B input stats: {'mean_chars': 147.14772727272728, 'max_chars': 992, 'p95_chars': 308.69999999999993, 'mean_tokens': 45.88920454545455, 'max_tokens': 235, 'p95_tokens': 90.44999999999999}
Task B target stats: {'mean_chars': 2415.9545454545455, 'max_chars': 5283, 'p95_chars': 3509.7499999999995, 'mean_tokens': 1268.0738636363637, 'max_tokens': 2977, 'p95_tokens': 1939.6999999999994}
TASK_A_MAX_TARGET_LEN: 529
TASK_B_MAX_TARGET_LEN: 896


In [14]:
_embedding_model = None

def get_embedding_model():
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))
    return _embedding_model

def embedding_cosine_similarity(text_a, text_b):
    model = get_embedding_model()
    emb = model.encode([text_a, text_b], convert_to_numpy=True, show_progress_bar=False)
    return float(cosine_similarity([emb[0]], [emb[1]])[0][0])

def check_balanced(text, open_sym, close_sym):
    depth = 0
    for ch in text:
        if ch == open_sym:
            depth += 1
        elif ch == close_sym:
            depth -= 1
            if depth < 0:
                return False
    return depth == 0

def structural_score_st(text):
    if not text or len(text.strip()) == 0:
        return 0.0
    upper = text.upper()
    checks = []
    checks.append("PROGRAM" in upper and "END_PROGRAM" in upper)
    checks.append("VAR" in upper and "END_VAR" in upper)
    if "IF" in upper:
        checks.append("END_IF" in upper)
    if "CASE" in upper:
        checks.append("END_CASE" in upper)
    if "FOR" in upper:
        checks.append("END_FOR" in upper)
    if "WHILE" in upper:
        checks.append("END_WHILE" in upper)
    checks.append(":=" in text)
    checks.append(check_balanced(text, "(", ")"))
    return sum(checks) / len(checks) if checks else 0.0

def structural_score_rust(text):
    if not text or len(text.strip()) == 0:
        return 0.0
    checks = []
    checks.append(check_balanced(text, "{", "}"))
    checks.append(check_balanced(text, "(", ")"))
    checks.append("fn " in text)
    checks.append("let " in text or "struct " in text or "impl " in text)
    if "if " in text:
        checks.append(True)
    if "match " in text:
        checks.append(True)
    return sum(checks) / len(checks) if checks else 0.0

def syntax_valid_st(text):
    if not text:
        return False
    upper = text.upper()
    return ("PROGRAM" in upper and "END_PROGRAM" in upper) and check_balanced(text, "(", ")")

def syntax_valid_rust(text):
    if not text:
        return False
    return check_balanced(text, "{", "}") and ("fn " in text)

def clean_generated_code(text, task):
    if text is None:
        return ""
    text = text.strip()
    text = re.sub(r"```[a-zA-Z]*\n?", "", text)
    text = text.replace("```", "")
    lines = text.split("\n")
    filtered_lines = []
    skip_prefixes = ("here is", "here's", "sure,", "sure!", "explanation:", "note:", "the following")
    for line in lines:
        stripped_lower = line.strip().lower()
        if any(stripped_lower.startswith(p) for p in skip_prefixes):
            continue
        filtered_lines.append(line)
    return "\n".join(filtered_lines).strip()

def score_candidate(candidate_text, task, reference_text=None):
    if not candidate_text or len(candidate_text.strip()) == 0:
        return -1.0
    if task == "A":
        struct_score = structural_score_st(candidate_text)
        valid = syntax_valid_st(candidate_text)
    else:
        struct_score = structural_score_rust(candidate_text)
        valid = syntax_valid_rust(candidate_text)
    validity_score = 1.0 if valid else 0.3
    combined = 0.5 * struct_score + 0.5 * validity_score
    if reference_text is not None:
        try:
            sem_sim = embedding_cosine_similarity(candidate_text, reference_text)
        except Exception:
            sem_sim = 0.0
        combined = 0.3 * struct_score + 0.3 * validity_score + 0.4 * sem_sim
    return combined

def select_best_candidate(candidates, task, reference_text=None):
    best_candidate, best_score = None, -999.0
    for c in candidates:
        score = score_candidate(c["text"], task, reference_text)
        if score > best_score:
            best_score, best_candidate = score, c
    return best_candidate, best_score

In [15]:
TASK_A_PROMPTS = {
    "v1_direct": (
        "You are an expert PLC programmer. Convert the following natural language "
        "description of industrial control logic into valid IEC 61131-3 Structured Text. "
        "Preserve all conditions, assignments, timers, loops, and variables exactly as implied. "
        "Output only the Structured Text code with no explanation.\n\n"
        "Description: {input}\n\nStructured Text:\n"
    ),
    "v2_structured": (
        "Task: Natural language to IEC 61131-3 Structured Text.\n"
        "Requirements: syntactically valid ST, correct PROGRAM/END_PROGRAM and VAR/END_VAR blocks, "
        "correct IF/END_IF, CASE/END_CASE, FOR/END_FOR, WHILE/END_WHILE where applicable, correct timers, "
        "deterministic and safety-preserving logic. Do not repeat variable declarations. "
        "Output code only, no markdown, no explanation.\n\n"
        "Input:\n{input}\n\nOutput:\n"
    ),
    "v3_fewshot": (
        "Example.\n"
        "Description: Start the conveyor belt when 'Start' Button is pressed and upon release "
        "the start button the conveyor maintains the running state. 'Stop' Button is used to stop the conveyor belt.\n"
        "Structured Text:\n"
        "PROGRAM start_stop\nVAR\nstart AT %IX0.0 : BOOL;\nstop AT %IX0.1 : BOOL;\nLED AT %QX0.0 : BOOL;\nEND_VAR\n\n"
        "LED := stop AND (LED OR start);\nEND_PROGRAM\n\n"
        "Now convert this new description into Structured Text using the same style. "
        "Output only code.\n\nDescription: {input}\n\nStructured Text:\n"
    )
}

TASK_B_PROMPTS = {
    "v1_direct": (
        "You are an expert Rust systems programmer specializing in industrial control systems. "
        "Convert the following natural language description of industrial control logic into a Rust program. "
        "Preserve all conditions, state handling, and control flow. "
        "Output only the Rust code with no explanation.\n\n"
        "Description: {input}\n\nRust code:\n"
    ),
    "v2_structured": (
        "Task: Natural language to Rust implementation of deterministic industrial control logic.\n"
        "Requirements: syntactically valid Rust, correct structs/functions representing PLC state, "
        "correct conditions and control flow, no unnecessary networking or Modbus code unless implied, "
        "no hallucinated hardware addresses. Output code only, no markdown, no explanation.\n\n"
        "Input:\n{input}\n\nOutput:\n"
    ),
    "v3_fewshot": (
        "Example.\n"
        "Description: Start the conveyor belt when Start button is pressed, latch running state, "
        "stop when Stop button is pressed.\n"
        "Rust code:\n"
        "struct Conveyor {{ running: bool }}\n\n"
        "impl Conveyor {{\n"
        "    fn update(&mut self, start: bool, stop: bool) {{\n"
        "        self.running = stop == false && (self.running || start);\n"
        "    }}\n"
        "}}\n\n"
        "fn main() {{\n"
        "    let mut conveyor = Conveyor {{ running: false }};\n"
        "}}\n\n"
        "Now convert this new description into Rust using a similar style. "
        "Output only code.\n\nDescription: {input}\n\nRust code:\n"
    )
}

def build_prompt(task, input_text, prompt_version):
    template = TASK_A_PROMPTS[prompt_version] if task == "A" else TASK_B_PROMPTS[prompt_version]
    return template.format(input=input_text)

In [16]:
MODEL_REGISTRY = {
    "qwen_coder_1_5b": {"repo": "Qwen/Qwen2.5-Coder-1.5B-Instruct", "params_b": 1.5},
    "qwen_coder_3b": {"repo": "Qwen/Qwen2.5-Coder-3B-Instruct", "params_b": 3.0},
    "qwen_coder_7b": {"repo": "Qwen/Qwen2.5-Coder-7B-Instruct", "params_b": 7.0},
    "deepseek_coder_1_3b": {"repo": "deepseek-ai/deepseek-coder-1.3b-instruct", "params_b": 1.3},
    "deepseek_coder_6_7b": {"repo": "deepseek-ai/deepseek-coder-6.7b-instruct", "params_b": 6.7}
}

for name, info in MODEL_REGISTRY.items():
    assert info["params_b"] <= 8.0, f"{name} exceeds 8B parameter restriction"

_loaded_model = None
_loaded_tokenizer = None
_loaded_model_name = None

def unload_current_model():
    global _loaded_model, _loaded_tokenizer, _loaded_model_name
    if _loaded_model is not None:
        del _loaded_model
    if _loaded_tokenizer is not None:
        del _loaded_tokenizer
    _loaded_model = None
    _loaded_tokenizer = None
    _loaded_model_name = None
    clear_gpu_memory()

def load_model_safe(model_key, quantization="4bit", use_lora_prep=False, max_retries=3, retry_delay=5):
    global _loaded_model, _loaded_tokenizer, _loaded_model_name

    if _loaded_model_name == model_key:
        return _loaded_model, _loaded_tokenizer

    unload_current_model()
    repo = MODEL_REGISTRY[model_key]["repo"]

    tokenizer = None
    for attempt in range(1, max_retries + 1):
        try:
            tokenizer = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
            break
        except Exception as e:
            print(f"Attempt {attempt}/{max_retries} failed to load tokenizer for {model_key}: {e}")
            if attempt < max_retries:
                time.sleep(retry_delay)

    if tokenizer is None:
        print(f"Could not load {model_key} after {max_retries} attempts. Skipping.")
        return None, None

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = None
    torch_dtype = torch.float16

    if quantization == "4bit" and torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4"
        )
    elif quantization == "8bit" and torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(load_in_8bit=True)

    model = None
    for attempt in range(1, max_retries + 1):
        try:
            if quant_config is not None:
                model = AutoModelForCausalLM.from_pretrained(
                    repo, quantization_config=quant_config, device_map="auto", trust_remote_code=True
                )
            else:
                model = AutoModelForCausalLM.from_pretrained(
                    repo, torch_dtype=torch_dtype, device_map="auto", trust_remote_code=True
                )
            break
        except torch.cuda.OutOfMemoryError:
            clear_gpu_memory()
            print(f"OOM while loading {model_key}. Retrying with 4bit.")
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4"
            )
        except Exception as e:
            print(f"Attempt {attempt}/{max_retries} failed to load model {model_key}: {e}")
            if attempt < max_retries:
                time.sleep(retry_delay)

    if model is None:
        print(f"Could not load model {model_key} after {max_retries} attempts. Skipping.")
        return None, None

    if use_lora_prep and quant_config is not None:
        model = prepare_model_for_kbit_training(model)

    model.gradient_checkpointing_enable()

    _loaded_model = model
    _loaded_tokenizer = tokenizer
    _loaded_model_name = model_key

    print_gpu_status(f"after loading {model_key}")
    return model, tokenizer

In [17]:
class StopOnSubstring(StoppingCriteria):
    def __init__(self, tokenizer, stop_strings, prompt_len):
        self.tokenizer = tokenizer
        self.stop_strings = stop_strings
        self.prompt_len = prompt_len

    def __call__(self, input_ids, scores, **kwargs):
        generated_ids = input_ids[0][self.prompt_len:]
        if len(generated_ids) < 5:
            return False
        decoded = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        return any(s in decoded for s in self.stop_strings)

def generate_with_fallback(model, tokenizer, prompt, max_new_tokens=512, temperature=0.2,
                            do_sample=False, num_beams=1, stop_strings=None):
    attempt_configs = [
        {"max_new_tokens": max_new_tokens},
        {"max_new_tokens": max(max_new_tokens // 2, 64)},
        {"max_new_tokens": max(max_new_tokens // 4, 32)}
    ]

    for cfg in attempt_configs:
        try:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
            prompt_len = inputs["input_ids"].shape[1]

            gen_kwargs = {
                "max_new_tokens": cfg["max_new_tokens"],
                "pad_token_id": tokenizer.pad_token_id,
                "eos_token_id": tokenizer.eos_token_id,
                "repetition_penalty": 1.15,
                "no_repeat_ngram_size": 4
            }
            if do_sample:
                gen_kwargs["do_sample"] = True
                gen_kwargs["temperature"] = temperature
                gen_kwargs["top_p"] = 0.9
            else:
                gen_kwargs["do_sample"] = False
                if num_beams > 1:
                    gen_kwargs["num_beams"] = num_beams

            if stop_strings:
                gen_kwargs["stopping_criteria"] = StoppingCriteriaList([
                    StopOnSubstring(tokenizer, stop_strings, prompt_len)
                ])

            with torch.no_grad():
                output_ids = model.generate(**inputs, **gen_kwargs)

            generated_text = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)
            clear_gpu_memory()
            return generated_text

        except torch.cuda.OutOfMemoryError:
            clear_gpu_memory()
            print(f"OOM at max_new_tokens={cfg['max_new_tokens']}. Retrying smaller.")
            continue
        except Exception as e:
            print(f"Generation error: {e}. Retrying smaller.")
            clear_gpu_memory()
            continue

    return ""

TASK_A_STOP_STRINGS = ["END_PROGRAM"]
TASK_B_STOP_STRINGS = ["\nfn main()", "std::thread::sleep"]

def get_stop_strings(task):
    return TASK_A_STOP_STRINGS if task == "A" else TASK_B_STOP_STRINGS

In [18]:
experiment_records = []

def log_experiment(name, model, task, prompt_version, quantization, lora_config,
                    learning_rate, epochs, seq_length, val_loss, gen_quality,
                    syntax_validity, structural_similarity, semantic_similarity,
                    inference_time, gpu_memory_gb):
    experiment_records.append({
        "experiment_name": name, "model": model, "task": task, "prompt_version": prompt_version,
        "quantization": quantization, "lora_config": lora_config, "learning_rate": learning_rate,
        "epochs": epochs, "seq_length": seq_length, "val_loss": val_loss, "gen_quality": gen_quality,
        "syntax_validity": syntax_validity, "structural_similarity": structural_similarity,
        "semantic_similarity": semantic_similarity, "inference_time": inference_time,
        "gpu_memory_gb": gpu_memory_gb
    })

def get_experiment_table():
    return pd.DataFrame(experiment_records)

In [19]:
def evaluate_on_validation(model_key, task, prompt_version, val_df, quantization="4bit",
                            max_examples=30, max_new_tokens=384):
    if len(val_df) == 0:
        print(f"No validation data available for task {task}. Skipping.")
        return None

    model, tokenizer = load_model_safe(model_key, quantization=quantization)
    if model is None:
        print(f"Skipping evaluation for {model_key} on task {task}: model unavailable.")
        return None

    subset = val_df.head(max_examples).reset_index(drop=True)
    struct_scores, syntax_scores, semantic_scores = [], [], []
    start_time = time.time()

    for idx, row in subset.iterrows():
        prompt = build_prompt(task, row["input"], prompt_version)
        raw_output = generate_with_fallback(model, tokenizer, prompt, max_new_tokens=max_new_tokens,
                                             stop_strings=get_stop_strings(task))
        cleaned = clean_generated_code(raw_output, task)

        if task == "A":
            struct_scores.append(structural_score_st(cleaned))
            syntax_scores.append(1.0 if syntax_valid_st(cleaned) else 0.0)
        else:
            struct_scores.append(structural_score_rust(cleaned))
            syntax_scores.append(1.0 if syntax_valid_rust(cleaned) else 0.0)

        try:
            sem = embedding_cosine_similarity(cleaned, row["target"]) if len(cleaned) > 0 else 0.0
        except Exception:
            sem = 0.0
        semantic_scores.append(sem)

    elapsed = time.time() - start_time
    avg_time = elapsed / max(len(subset), 1)
    gpu_mem = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0

    return {
        "structural_similarity": float(np.mean(struct_scores)),
        "syntax_validity": float(np.mean(syntax_scores)),
        "semantic_similarity": float(np.mean(semantic_scores)),
        "inference_time": avg_time,
        "gpu_memory_gb": gpu_mem
    }

In [20]:
baseline1_model = "qwen_coder_1_5b"

if len(task_a_val) > 0:
    result_a_baseline1 = evaluate_on_validation(baseline1_model, "A", "v2_structured", task_a_val, quantization="4bit")
    if result_a_baseline1:
        log_experiment("baseline1_task_a", baseline1_model, "A", "v2_structured", "4bit", None,
                        None, None, TASK_A_MAX_TARGET_LEN, None,
                        result_a_baseline1["structural_similarity"], result_a_baseline1["syntax_validity"],
                        result_a_baseline1["structural_similarity"], result_a_baseline1["semantic_similarity"],
                        result_a_baseline1["inference_time"], result_a_baseline1["gpu_memory_gb"])
        print("Baseline 1 Task A:", result_a_baseline1)

if len(task_b_val) > 0:
    result_b_baseline1 = evaluate_on_validation(baseline1_model, "B", "v2_structured", task_b_val, quantization="4bit")
    if result_b_baseline1:
        log_experiment("baseline1_task_b", baseline1_model, "B", "v2_structured", "4bit", None,
                        None, None, TASK_B_MAX_TARGET_LEN, None,
                        result_b_baseline1["structural_similarity"], result_b_baseline1["syntax_validity"],
                        result_b_baseline1["structural_similarity"], result_b_baseline1["semantic_similarity"],
                        result_b_baseline1["inference_time"], result_b_baseline1["gpu_memory_gb"])
        print("Baseline 1 Task B:", result_b_baseline1)

unload_current_model()

[after loading qwen_coder_1_5b] Allocated: 1.07 GB | Reserved: 1.09 GB | Total: 14.84 GB


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

Baseline 1 Task A: {'structural_similarity': 0.8211507936507937, 'syntax_validity': 0.6333333333333333, 'semantic_similarity': 0.6244653711716334, 'inference_time': 11.486517548561096, 'gpu_memory_gb': 1.166717529296875}


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

Baseline 1 Task B: {'structural_similarity': 0.85, 'syntax_validity': 0.9333333333333333, 'semantic_similarity': 0.24748710642258326, 'inference_time': 7.165675067901612, 'gpu_memory_gb': 1.166717529296875}


In [21]:
baseline2_model = "qwen_coder_7b"

if len(task_a_val) > 0:
    result_a_baseline2 = evaluate_on_validation(baseline2_model, "A", "v2_structured", task_a_val, quantization="4bit")
    if result_a_baseline2:
        log_experiment("baseline2_task_a", baseline2_model, "A", "v2_structured", "4bit", None,
                        None, None, TASK_A_MAX_TARGET_LEN, None,
                        result_a_baseline2["structural_similarity"], result_a_baseline2["syntax_validity"],
                        result_a_baseline2["structural_similarity"], result_a_baseline2["semantic_similarity"],
                        result_a_baseline2["inference_time"], result_a_baseline2["gpu_memory_gb"])
        print("Baseline 2 Task A:", result_a_baseline2)

if len(task_b_val) > 0:
    result_b_baseline2 = evaluate_on_validation(baseline2_model, "B", "v2_structured", task_b_val, quantization="4bit")
    if result_b_baseline2:
        log_experiment("baseline2_task_b", baseline2_model, "B", "v2_structured", "4bit", None,
                        None, None, TASK_B_MAX_TARGET_LEN, None,
                        result_b_baseline2["structural_similarity"], result_b_baseline2["syntax_validity"],
                        result_b_baseline2["structural_similarity"], result_b_baseline2["semantic_similarity"],
                        result_b_baseline2["inference_time"], result_b_baseline2["gpu_memory_gb"])
        print("Baseline 2 Task B:", result_b_baseline2)

unload_current_model()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[after loading qwen_coder_7b] Allocated: 5.31 GB | Reserved: 5.36 GB | Total: 14.84 GB


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

Baseline 2 Task A: {'structural_similarity': 0.28777777777777774, 'syntax_validity': 0.26666666666666666, 'semantic_similarity': 0.20779497027397156, 'inference_time': 2.9736316998799643, 'gpu_memory_gb': 5.310746192932129}


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

Baseline 2 Task B: {'structural_similarity': 0.9666666666666667, 'syntax_validity': 0.8666666666666667, 'semantic_similarity': 0.35374105473359424, 'inference_time': 11.518551874160767, 'gpu_memory_gb': 5.310746192932129}


In [22]:
prompt_versions = ["v1_direct", "v2_structured", "v3_fewshot"]
prompt_comparison_results = {"A": {}, "B": {}}

if len(task_a_val) > 0:
    for pv in prompt_versions:
        res = evaluate_on_validation(baseline1_model, "A", pv, task_a_val, quantization="4bit", max_examples=20)
        prompt_comparison_results["A"][pv] = res
        print(f"Task A prompt {pv}:", res)
    unload_current_model()

if len(task_b_val) > 0:
    for pv in prompt_versions:
        res = evaluate_on_validation(baseline1_model, "B", pv, task_b_val, quantization="4bit", max_examples=20)
        prompt_comparison_results["B"][pv] = res
        print(f"Task B prompt {pv}:", res)
    unload_current_model()

def best_prompt_for_task(task):
    results = prompt_comparison_results[task]
    valid_results = {k: v for k, v in results.items() if v is not None}
    if not valid_results:
        return "v2_structured"
    def combined_score(r):
        return 0.4 * r["structural_similarity"] + 0.3 * r["syntax_validity"] + 0.3 * r["semantic_similarity"]
    return max(valid_results, key=lambda k: combined_score(valid_results[k]))

BEST_PROMPT_TASK_A = best_prompt_for_task("A")
BEST_PROMPT_TASK_B = best_prompt_for_task("B")

print("Best prompt Task A:", BEST_PROMPT_TASK_A)
print("Best prompt Task B:", BEST_PROMPT_TASK_B)

[after loading qwen_coder_1_5b] Allocated: 1.17 GB | Reserved: 1.19 GB | Total: 14.84 GB


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

Task A prompt v1_direct: {'structural_similarity': 0.7219047619047618, 'syntax_validity': 0.3, 'semantic_similarity': 0.5883227780461311, 'inference_time': 20.277844429016113, 'gpu_memory_gb': 1.1677331924438477}


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

Task A prompt v2_structured: {'structural_similarity': 0.7926190476190477, 'syntax_validity': 0.55, 'semantic_similarity': 0.6144994631409645, 'inference_time': 11.916406643390655, 'gpu_memory_gb': 1.1677331924438477}


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

... (3 outputs left out)

c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

Task B prompt v2_structured: {'structural_similarity': 0.8625, 'syntax_validity': 0.95, 'semantic_similarity': 0.2412310555577278, 'inference_time': 6.245132780075073, 'gpu_memory_gb': 1.1677331924438477}


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\trans

Task B prompt v3_fewshot: {'structural_similarity': 0.7825, 'syntax_validity': 0.45, 'semantic_similarity': 0.25649702474474906, 'inference_time': 12.495577490329742, 'gpu_memory_gb': 1.1677331924438477}
Best prompt Task A: v3_fewshot
Best prompt Task B: v2_structured


In [23]:
def build_finetune_examples(df, task, prompt_version, tokenizer):
    examples = []
    for _, row in df.iterrows():
        prompt = build_prompt(task, row["input"], prompt_version)
        target = row["target"] + tokenizer.eos_token
        examples.append({"text": prompt + target, "prompt": prompt, "target": target})
    return examples

def tokenize_finetune_example(example, tokenizer, max_length):
    tokenized = tokenizer(example["text"], truncation=True, max_length=max_length, padding="max_length")
    prompt_ids = tokenizer(example["prompt"], truncation=True, max_length=max_length)["input_ids"]
    labels = list(tokenized["input_ids"])
    prompt_len = min(len(prompt_ids), max_length)
    for i in range(prompt_len):
        labels[i] = -100
    for i in range(len(labels)):
        if tokenized["attention_mask"][i] == 0:
            labels[i] = -100
    tokenized["labels"] = labels
    return tokenized

class FinetuneDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        tok = tokenize_finetune_example(self.examples[idx], self.tokenizer, self.max_length)
        return {
            "input_ids": torch.tensor(tok["input_ids"]),
            "attention_mask": torch.tensor(tok["attention_mask"]),
            "labels": torch.tensor(tok["labels"])
        }

In [24]:
MIN_EXAMPLES_FOR_FINETUNE = 30

def run_lora_finetune(model_key, task, train_data, val_data, prompt_version,
                       max_input_len, max_target_len, lora_r=16, lora_alpha=32,
                       lora_dropout=0.05, learning_rate=2e-4, epochs=2,
                       batch_size=1, grad_accum=8):

    if len(train_data) < MIN_EXAMPLES_FOR_FINETUNE:
        print(f"Not enough training examples for task {task} ({len(train_data)}). Skipping.")
        return None

    model, tokenizer = load_model_safe(model_key, quantization="4bit", use_lora_prep=True)
    if model is None:
        return None

    lora_config = LoraConfig(
        r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout, bias="none",
        task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()

    max_length = max_input_len + max_target_len
    train_examples = build_finetune_examples(train_data, task, prompt_version, tokenizer)
    train_dataset = FinetuneDataset(train_examples, tokenizer, max_length)

    output_dir = os.path.join(CHECKPOINT_DIR, f"lora_{model_key}_task{task}")

    training_args = TrainingArguments(
        output_dir=output_dir, per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum, num_train_epochs=epochs,
        learning_rate=learning_rate, warmup_ratio=0.06, weight_decay=0.01,
        fp16=True, logging_steps=10, save_strategy="epoch", save_total_limit=1,
        report_to=[], gradient_checkpointing=True, optim="paged_adamw_8bit"
    )

    try:
        trainer = Trainer(model=peft_model, args=training_args, train_dataset=train_dataset)
        trainer.train()
    except torch.cuda.OutOfMemoryError:
        clear_gpu_memory()
        print("OOM during fine-tuning. Retrying with smaller batch size.")
        training_args.per_device_train_batch_size = 1
        training_args.gradient_accumulation_steps = grad_accum * 2
        trainer = Trainer(model=peft_model, args=training_args, train_dataset=train_dataset)
        trainer.train()

    final_val_loss = None
    if trainer.state.log_history:
        losses = [e["loss"] for e in trainer.state.log_history if "loss" in e]
        if losses:
            final_val_loss = losses[-1]

    peft_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    eval_result = None
    if len(val_data) > 0:
        subset = val_data.head(20).reset_index(drop=True)
        struct_scores, syntax_scores, semantic_scores = [], [], []
        for _, row in subset.iterrows():
            prompt = build_prompt(task, row["input"], prompt_version)
            raw = generate_with_fallback(peft_model, tokenizer, prompt, max_new_tokens=max_target_len,
                                          stop_strings=get_stop_strings(task))
            cleaned = clean_generated_code(raw, task)
            if task == "A":
                struct_scores.append(structural_score_st(cleaned))
                syntax_scores.append(1.0 if syntax_valid_st(cleaned) else 0.0)
            else:
                struct_scores.append(structural_score_rust(cleaned))
                syntax_scores.append(1.0 if syntax_valid_rust(cleaned) else 0.0)
            try:
                sem = embedding_cosine_similarity(cleaned, row["target"]) if len(cleaned) > 0 else 0.0
            except Exception:
                sem = 0.0
            semantic_scores.append(sem)

        eval_result = {
            "structural_similarity": float(np.mean(struct_scores)),
            "syntax_validity": float(np.mean(syntax_scores)),
            "semantic_similarity": float(np.mean(semantic_scores))
        }

    log_experiment(
        f"baseline3_lora_task_{task}", model_key, task, prompt_version, "4bit",
        f"r={lora_r},alpha={lora_alpha},dropout={lora_dropout}", learning_rate, epochs, max_length,
        final_val_loss,
        eval_result["structural_similarity"] if eval_result else None,
        eval_result["syntax_validity"] if eval_result else None,
        eval_result["structural_similarity"] if eval_result else None,
        eval_result["semantic_similarity"] if eval_result else None,
        None, torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
    )

    del trainer
    unload_current_model()
    return {"output_dir": output_dir, "val_loss": final_val_loss, "eval": eval_result}

lora_result_a = run_lora_finetune(
    baseline1_model, "A", task_a_train, task_a_val, "v3_fewshot",
    TASK_A_MAX_INPUT_LEN, TASK_A_MAX_TARGET_LEN
) if len(task_a_train) >= MIN_EXAMPLES_FOR_FINETUNE else None

lora_result_b = run_lora_finetune(
    baseline1_model, "B", task_b_train, task_b_val, "v2_structured",
    TASK_B_MAX_INPUT_LEN, TASK_B_MAX_TARGET_LEN
) if len(task_b_train) >= MIN_EXAMPLES_FOR_FINETUNE else None

print("LoRA Task A result:", lora_result_a)
print("LoRA Task B result:", lora_result_b)

[after loading qwen_coder_1_5b] Allocated: 1.60 GB | Reserved: 2.06 GB | Total: 14.84 GB
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/74 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


{'loss': 1.3814, 'grad_norm': 1.0572694540023804, 'learning_rate': 0.0001855072463768116, 'epoch': 0.27}
{'loss': 0.5896, 'grad_norm': 0.8550523519515991, 'learning_rate': 0.0001565217391304348, 'epoch': 0.54}
{'loss': 0.4335, 'grad_norm': 0.6843307018280029, 'learning_rate': 0.00012753623188405797, 'epoch': 0.8}
{'loss': 0.3159, 'grad_norm': 0.6349360346794128, 'learning_rate': 9.855072463768117e-05, 'epoch': 1.07}
{'loss': 0.2751, 'grad_norm': 0.6501144170761108, 'learning_rate': 6.956521739130436e-05, 'epoch': 1.34}
{'loss': 0.2584, 'grad_norm': 0.5667492747306824, 'learning_rate': 4.057971014492754e-05, 'epoch': 1.61}
{'loss': 0.2355, 'grad_norm': 0.46358522772789, 'learning_rate': 1.1594202898550725e-05, 'epoch': 1.87}
{'train_runtime': 456.6796, 'train_samples_per_second': 1.309, 'train_steps_per_second': 0.162, 'train_loss': 0.4870615649867702, 'epoch': 1.98}


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\torch

... (2 outputs left out)

  0%|          | 0/74 [00:00<?, ?it/s]

{'loss': 0.5268, 'grad_norm': 0.2915171682834625, 'learning_rate': 0.0001855072463768116, 'epoch': 0.27}
{'loss': 0.2759, 'grad_norm': 0.28272461891174316, 'learning_rate': 0.0001565217391304348, 'epoch': 0.54}
{'loss': 0.1819, 'grad_norm': 0.19078540802001953, 'learning_rate': 0.00012753623188405797, 'epoch': 0.8}
{'loss': 0.1526, 'grad_norm': 0.15549764037132263, 'learning_rate': 9.855072463768117e-05, 'epoch': 1.07}
{'loss': 0.1353, 'grad_norm': 0.14760519564151764, 'learning_rate': 6.956521739130436e-05, 'epoch': 1.34}
{'loss': 0.1368, 'grad_norm': 0.2176298201084137, 'learning_rate': 4.057971014492754e-05, 'epoch': 1.61}
{'loss': 0.1187, 'grad_norm': 0.19432471692562103, 'learning_rate': 1.1594202898550725e-05, 'epoch': 1.87}
{'train_runtime': 560.5179, 'train_samples_per_second': 1.067, 'train_steps_per_second': 0.132, 'train_loss': 0.21371111837593285, 'epoch': 1.98}


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\torch

LoRA Task A result: {'output_dir': 'C:/Users/VGST/Desktop/AI4Industrial Automation/checkpoints\\lora_qwen_coder_1_5b_taskA', 'val_loss': 0.2355, 'eval': {'structural_similarity': 0.8, 'syntax_validity': 0.8, 'semantic_similarity': 0.7694920688867569}}
LoRA Task B result: {'output_dir': 'C:/Users/VGST/Desktop/AI4Industrial Automation/checkpoints\\lora_qwen_coder_1_5b_taskB', 'val_loss': 0.1187, 'eval': {'structural_similarity': 0.515, 'syntax_validity': 0.0, 'semantic_similarity': 0.970542985200882}}


In [25]:
def generate_candidates_batched(input_texts, task, prompt_version, candidate_config_pool):
    num_examples = len(input_texts)
    candidates_per_example = [[] for _ in range(num_examples)]

    for cfg in candidate_config_pool:
        model_key = cfg["model_key"]
        quant = cfg.get("quantization", "4bit")
        do_sample = cfg.get("do_sample", False)
        temperature = cfg.get("temperature", 0.2)
        lora_path = cfg.get("lora_path", None)

        model, tokenizer = load_model_safe(model_key, quantization=quant)
        if model is None:
            print(f"Skipping candidate config {cfg}: model unavailable.")
            continue

        baseline_gb = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0

        active_model = model
        if lora_path is not None:
            active_model = PeftModel.from_pretrained(model, lora_path)

        for i, input_text in enumerate(input_texts):
            prompt = build_prompt(task, input_text, prompt_version)
            raw_output = generate_with_fallback(
                active_model, tokenizer, prompt, max_new_tokens=768,
                temperature=temperature, do_sample=do_sample, stop_strings=get_stop_strings(task)
            )
            cleaned = clean_generated_code(raw_output, task)
            candidates_per_example[i].append({"config": cfg, "text": cleaned})

        if lora_path is not None:
            del active_model
        clear_gpu_memory()
        assert_memory_released(
            f"after candidate config {model_key} lora={lora_path is not None}",
            baseline_gb=baseline_gb if lora_path is None else 0.0
        )

    return candidates_per_example

In [26]:
experiment_table = get_experiment_table()
print("Full experiment table:")
print(experiment_table.to_string())

def rank_experiments(df, task):
    task_df = df[df["task"] == task].copy()
    if len(task_df) == 0:
        return task_df
    def rank_score(row):
        struct = row["structural_similarity"] if pd.notnull(row["structural_similarity"]) else 0
        syntax = row["syntax_validity"] if pd.notnull(row["syntax_validity"]) else 0
        semantic = row["semantic_similarity"] if pd.notnull(row["semantic_similarity"]) else 0
        return 0.4 * struct + 0.3 * syntax + 0.3 * semantic
    task_df["rank_score"] = task_df.apply(rank_score, axis=1)
    return task_df.sort_values("rank_score", ascending=False).reset_index(drop=True)

leaderboard_a = rank_experiments(experiment_table, "A")
leaderboard_b = rank_experiments(experiment_table, "B")

print("\nTask A leaderboard:")
print(leaderboard_a.to_string())
print("\nTask B leaderboard:")
print(leaderboard_b.to_string())

def get_best_experiment(leaderboard):
    return leaderboard.iloc[0].to_dict() if len(leaderboard) > 0 else None

best_task_a_experiment = get_best_experiment(leaderboard_a)
best_task_b_experiment = get_best_experiment(leaderboard_b)

print("\nBest Task A strategy:", best_task_a_experiment)
print("Best Task B strategy:", best_task_b_experiment)

Full experiment table:
         experiment_name            model task prompt_version quantization                 lora_config  learning_rate  epochs  seq_length  val_loss  gen_quality  syntax_validity  structural_similarity  semantic_similarity  inference_time  gpu_memory_gb
0       baseline1_task_a  qwen_coder_1_5b    A  v2_structured         4bit                        None            NaN     NaN         529       NaN     0.821151         0.633333               0.821151             0.624465       11.486518       1.166718
1       baseline1_task_b  qwen_coder_1_5b    B  v2_structured         4bit                        None            NaN     NaN         896       NaN     0.850000         0.933333               0.850000             0.247487        7.165675       1.166718
2       baseline2_task_a    qwen_coder_7b    A  v2_structured         4bit                        None            NaN     NaN         529       NaN     0.287778         0.266667               0.287778             0.207

In [27]:
test_df["_clean_input"] = test_df["TASK"].apply(clean_text_field)
test_df["_row_id"] = test_df.index

missing_inputs = test_df["_clean_input"].isnull().sum()
print("Test rows total:", len(test_df))
print("Test rows with missing input text:", missing_inputs)

Test rows total: 30
Test rows with missing input text: 0


In [28]:
def save_predictions_cache(predictions, run_name):
    path = os.path.join(PREDICTIONS_CACHE_DIR, f"{run_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2)
    print(f"Cached {run_name} to {path}")

def load_predictions_cache(run_name):
    path = os.path.join(PREDICTIONS_CACHE_DIR, f"{run_name}.json")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            predictions = json.load(f)
        print(f"Loaded {run_name} from cache ({len(predictions)} predictions).")
        return predictions
    return None

def run_with_cache(run_name, generation_fn):
    cached = load_predictions_cache(run_name)
    if cached is not None:
        return cached
    predictions = generation_fn()
    save_predictions_cache(predictions, run_name)
    return predictions

def evict_failed_rows(run_name, failed_ids):
    final_path = os.path.join(PREDICTIONS_CACHE_DIR, f"{run_name}.json")
    partial_path = os.path.join(PREDICTIONS_CACHE_DIR, f"{run_name}_partial.json")

    if not os.path.exists(final_path):
        print(f"No final cache for {run_name} to evict from.")
        return

    with open(final_path, "r", encoding="utf-8") as f:
        final_predictions = json.load(f)

    partial_dict = {}
    for p in final_predictions:
        rid_key = str(p["row_id"])
        if p["row_id"] in failed_ids:
            partial_dict[rid_key] = {"row_id": p["row_id"], "RUST-CODE": p.get("RUST-CODE", "")}
        else:
            partial_dict[rid_key] = p

    with open(partial_path, "w", encoding="utf-8") as f:
        json.dump(partial_dict, f, indent=2)

    os.remove(final_path)
    print(f"Evicted {len(failed_ids)} rows from {run_name}. Ready to resume.")

In [29]:
def run_strategy_over_test(model_key, prompt_version_a, prompt_version_b, quantization,
                            do_sample, temperature, run_name,
                            lora_path_a=None, lora_path_b=None,
                            use_ensemble=False, candidate_config_pool_a=None, candidate_config_pool_b=None):

    test_inputs = test_df["_clean_input"].fillna("").tolist()
    row_ids = test_df["_row_id"].tolist()

    partial_path = os.path.join(PREDICTIONS_CACHE_DIR, f"{run_name}_partial.json")

    def load_partial():
        if os.path.exists(partial_path):
            with open(partial_path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    def save_partial(partial_dict):
        with open(partial_path, "w", encoding="utf-8") as f:
            json.dump(partial_dict, f, indent=2)

    partial = load_partial()
    if partial:
        print(f"Resuming {run_name}: {len(partial)}/{len(row_ids)} rows already done.")

    if use_ensemble:
        if candidate_config_pool_a is None or candidate_config_pool_b is None:
            raise ValueError("use_ensemble requires candidate_config_pool_a and candidate_config_pool_b")

        remaining_indices = [i for i, rid in enumerate(row_ids)
                              if str(rid) not in partial or "ST-CODE" not in partial[str(rid)]
                              or "RUST-CODE" not in partial[str(rid)]]

        if remaining_indices:
            remaining_inputs = [test_inputs[i] for i in remaining_indices]
            candidates_a = generate_candidates_batched(remaining_inputs, "A", prompt_version_a, candidate_config_pool_a)
            candidates_b = generate_candidates_batched(remaining_inputs, "B", prompt_version_b, candidate_config_pool_b)

            for j, i in enumerate(remaining_indices):
                best_a, _ = select_best_candidate(candidates_a[j], "A", reference_text=None)
                best_b, _ = select_best_candidate(candidates_b[j], "B", reference_text=None)
                partial[str(row_ids[i])] = {
                    "row_id": row_ids[i],
                    "ST-CODE": best_a["text"] if best_a else "",
                    "RUST-CODE": best_b["text"] if best_b else ""
                }
                save_partial(partial)

        unload_current_model()
        return [partial[str(rid)] for rid in row_ids]

    model, tokenizer = load_model_safe(model_key, quantization=quantization)
    if model is None:
        print(f"Model {model_key} unavailable. Returning empty predictions.")
        return [{"row_id": rid, "ST-CODE": "", "RUST-CODE": ""} for rid in row_ids]

    model_a = model
    if lora_path_a is not None:
        model_a = PeftModel.from_pretrained(model, lora_path_a)

    for i, input_text in enumerate(test_inputs):
        rid_key = str(row_ids[i])
        if rid_key in partial and "ST-CODE" in partial[rid_key]:
            continue
        prompt_a = build_prompt("A", input_text, prompt_version_a)
        raw_a = generate_with_fallback(model_a, tokenizer, prompt_a, max_new_tokens=768,
                                        do_sample=do_sample, temperature=temperature,
                                        stop_strings=get_stop_strings("A"))
        st_pred = clean_generated_code(raw_a, "A")
        partial.setdefault(rid_key, {"row_id": row_ids[i]})
        partial[rid_key]["ST-CODE"] = st_pred
        save_partial(partial)

    if lora_path_a is not None:
        del model_a
        clear_gpu_memory()

    model, tokenizer = load_model_safe(model_key, quantization=quantization)
    model_b = model
    if lora_path_b is not None and model is not None:
        model_b = PeftModel.from_pretrained(model, lora_path_b)

    for i, input_text in enumerate(test_inputs):
        rid_key = str(row_ids[i])
        if rid_key in partial and "RUST-CODE" in partial[rid_key]:
            continue
        if model is None:
            partial.setdefault(rid_key, {"row_id": row_ids[i]})
            partial[rid_key]["RUST-CODE"] = ""
            save_partial(partial)
            continue
        prompt_b = build_prompt("B", input_text, prompt_version_b)
        raw_b = generate_with_fallback(model_b, tokenizer, prompt_b, max_new_tokens=768,
                                        do_sample=do_sample, temperature=temperature,
                                        stop_strings=get_stop_strings("B"))
        rust_pred = clean_generated_code(raw_b, "B")
        partial.setdefault(rid_key, {"row_id": row_ids[i]})
        partial[rid_key]["RUST-CODE"] = rust_pred
        save_partial(partial)

    if lora_path_b is not None and model is not None:
        del model_b
        clear_gpu_memory()

    unload_current_model()
    return [partial[str(rid)] for rid in row_ids]

In [30]:
run1_predictions = run_with_cache(
    "run1_predictions",
    lambda: run_strategy_over_test(
        baseline1_model, BEST_PROMPT_TASK_A, BEST_PROMPT_TASK_B, "4bit",
        do_sample=False, temperature=0.0, run_name="run1_predictions",
        lora_path_a=lora_result_a["output_dir"] if lora_result_a else None,
        lora_path_b=lora_result_b["output_dir"] if lora_result_b else None
    )
)
print("Run 1 predictions:", len(run1_predictions))

run2_predictions = run_with_cache(
    "run2_predictions",
    lambda: run_strategy_over_test(
        baseline2_model, "v3_fewshot", "v3_fewshot", "4bit",
        do_sample=True, temperature=0.3, run_name="run2_predictions"
    )
)
print("Run 2 predictions:", len(run2_predictions))

candidate_config_pool_a = [
    {"model_key": baseline1_model, "quantization": "4bit", "lora_path": lora_result_a["output_dir"]} if lora_result_a
    else {"model_key": baseline1_model, "quantization": "4bit", "do_sample": False},
    {"model_key": baseline2_model, "quantization": "4bit", "do_sample": False}
]
candidate_config_pool_b = [
    {"model_key": baseline1_model, "quantization": "4bit", "lora_path": lora_result_b["output_dir"]} if lora_result_b
    else {"model_key": baseline1_model, "quantization": "4bit", "do_sample": False},
    {"model_key": baseline2_model, "quantization": "4bit", "do_sample": False}
]

run3_predictions = run_with_cache(
    "run3_predictions",
    lambda: run_strategy_over_test(
        baseline1_model, BEST_PROMPT_TASK_A, BEST_PROMPT_TASK_B, "4bit",
        do_sample=False, temperature=0.0, run_name="run3_predictions",
        use_ensemble=True,
        candidate_config_pool_a=candidate_config_pool_a,
        candidate_config_pool_b=candidate_config_pool_b
    )
)
print("Run 3 predictions:", len(run3_predictions))

Loaded run1_predictions from cache (30 predictions).
Run 1 predictions: 30
Loaded run2_predictions from cache (30 predictions).
Run 2 predictions: 30
Loaded run3_predictions from cache (30 predictions).
Run 3 predictions: 30


In [31]:
def diagnose_st_failures(predictions, name, threshold=0.3):
    results = []
    for p in predictions:
        text = p["ST-CODE"]
        score = structural_score_st(text)
        results.append({"row_id": p["row_id"], "score": score, "length": len(text) if text else 0})
    diag_df = pd.DataFrame(results)
    failing = diag_df[diag_df["score"] <= threshold]
    print(f"{name}: {len(failing)}/{len(diag_df)} rows failing structural check")
    if len(failing) > 0:
        print(failing.to_string())
    return failing["row_id"].tolist()

failing_run1 = diagnose_st_failures(run1_predictions, "Run 1")
failing_run2 = diagnose_st_failures(run2_predictions, "Run 2")
failing_run3 = diagnose_st_failures(run3_predictions, "Run 3")

if failing_run1:
    evict_failed_rows("run1_predictions", failing_run1)
    run1_predictions = run_with_cache(
        "run1_predictions",
        lambda: run_strategy_over_test(
            baseline1_model, BEST_PROMPT_TASK_A, BEST_PROMPT_TASK_B, "4bit",
            do_sample=False, temperature=0.0, run_name="run1_predictions",
            lora_path_a=lora_result_a["output_dir"] if lora_result_a else None,
            lora_path_b=lora_result_b["output_dir"] if lora_result_b else None
        )
    )

if failing_run2:
    evict_failed_rows("run2_predictions", failing_run2)
    run2_predictions = run_with_cache(
        "run2_predictions",
        lambda: run_strategy_over_test(
            baseline2_model, "v3_fewshot", "v3_fewshot", "4bit",
            do_sample=True, temperature=0.3, run_name="run2_predictions"
        )
    )

if failing_run3:
    evict_failed_rows("run3_predictions", failing_run3)
    run3_predictions = run_with_cache(
        "run3_predictions",
        lambda: run_strategy_over_test(
            baseline1_model, BEST_PROMPT_TASK_A, BEST_PROMPT_TASK_B, "4bit",
            do_sample=False, temperature=0.0, run_name="run3_predictions",
            use_ensemble=True,
            candidate_config_pool_a=candidate_config_pool_a,
            candidate_config_pool_b=candidate_config_pool_b
        )
    )

print("\nRe-checking after regeneration:")
diagnose_st_failures(run1_predictions, "Run 1")
diagnose_st_failures(run2_predictions, "Run 2")
diagnose_st_failures(run3_predictions, "Run 3")

Run 1: 5/30 rows failing structural check
    row_id  score  length
3        3   0.25    1670
9        9   0.25    1831
10      10   0.25    1650
22      22   0.25    1606
28      28   0.25    4212
Run 2: 13/30 rows failing structural check
    row_id  score  length
0        0   0.25       1
5        5   0.25       1
6        6   0.25     739
11      11   0.00       0
17      17   0.25       1
21      21   0.25       1
22      22   0.25     755
23      23   0.00       0
25      25   0.00       0
26      26   0.00       0
27      27   0.00       0
28      28   0.00       0
29      29   0.25     733
Run 3: 2/30 rows failing structural check
    row_id  score  length
22      22   0.25    1606
28      28   0.25    4212
Evicted 5 rows from run1_predictions. Ready to resume.
Resuming run1_predictions: 30/30 rows already done.
[after loading qwen_coder_1_5b] Allocated: 1.18 GB | Reserved: 1.20 GB | Total: 14.84 GB


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\peft\

Cached run1_predictions to C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\predictions_cache\run1_predictions.json
Evicted 13 rows from run2_predictions. Ready to resume.
Resuming run2_predictions: 30/30 rows already done.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[after loading qwen_coder_7b] Allocated: 5.32 GB | Reserved: 5.37 GB | Total: 14.84 GB
Cached run2_predictions to C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\predictions_cache\run2_predictions.json
Evicted 2 rows from run3_predictions. Ready to resume.
Resuming run3_predictions: 30/30 rows already done.
[after loading qwen_coder_1_5b] Allocated: 1.18 GB | Reserved: 1.20 GB | Total: 14.84 GB
WARNING at after candidate config qwen_coder_1_5b lora=True: 1.19 GB allocated (1.19 GB above baseline 0.00 GB).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[after loading qwen_coder_7b] Allocated: 6.41 GB | Reserved: 6.47 GB | Total: 14.84 GB
[after candidate config qwen_coder_7b lora=False] Allocated: 5.32 GB (expected)
[after loading qwen_coder_1_5b] Allocated: 1.18 GB | Reserved: 1.20 GB | Total: 14.84 GB
WARNING at after candidate config qwen_coder_1_5b lora=True: 1.19 GB allocated (1.19 GB above baseline 0.00 GB).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[after loading qwen_coder_7b] Allocated: 6.41 GB | Reserved: 6.47 GB | Total: 14.84 GB
[after candidate config qwen_coder_7b lora=False] Allocated: 5.32 GB (expected)
Cached run3_predictions to C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\predictions_cache\run3_predictions.json

Re-checking after regeneration:
Run 1: 0/30 rows failing structural check
Run 2: 11/30 rows failing structural check
    row_id  score  length
0        0    0.0       0
6        6    0.0       0
11      11    0.0       0
17      17    0.0       0
21      21    0.0       0
22      22    0.0       0
25      25    0.0       0
26      26    0.0       0
27      27    0.0       0
28      28    0.0       0
29      29    0.0       0
Run 3: 0/30 rows failing structural check


[]

In [32]:
def build_submission_dataframe(predictions):
    df = pd.DataFrame(predictions)
    df = df.rename(columns={"row_id": "ID"})
    return df[["ID", "ST-CODE", "RUST-CODE"]]

submission_run1 = build_submission_dataframe(run1_predictions)
submission_run2 = build_submission_dataframe(run2_predictions)
submission_run3 = build_submission_dataframe(run3_predictions)

def save_submission_csv_and_xlsx(df, base_name):
    csv_path = os.path.join(OUTPUT_DIR, f"{base_name}.csv")
    xlsx_path = os.path.join(OUTPUT_DIR, f"{base_name}.xlsx")
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    df.to_excel(xlsx_path, index=False, engine="openpyxl")
    print(f"Saved: {xlsx_path}")
    return csv_path, xlsx_path

for df, name in [(submission_run1, "run1"), (submission_run2, "run2"), (submission_run3, "run3")]:
    max_len = max(df["ST-CODE"].str.len().max(), df["RUST-CODE"].str.len().max())
    if max_len > 32767:
        print(f"WARNING: {name} contains a cell exceeding Excel's 32767 char limit ({max_len}). XLSX will truncate; CSV will not.")

submission_run1_csv_path, submission_run1_xlsx_path = save_submission_csv_and_xlsx(submission_run1, "submission_run1")
submission_run2_csv_path, submission_run2_xlsx_path = save_submission_csv_and_xlsx(submission_run2, "submission_run2")
submission_run3_csv_path, submission_run3_xlsx_path = save_submission_csv_and_xlsx(submission_run3, "submission_run3")

Saved: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_run1.csv
Saved: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_run1.xlsx
Saved: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_run2.csv
Saved: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_run2.xlsx
Saved: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_run3.csv
Saved: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_run3.xlsx


In [33]:
def validate_submission(df, name, expected_row_count):
    print("=" * 80)
    print(f"VALIDATION REPORT: {name}")
    print("=" * 80)

    checks = {}
    checks["correct_row_count"] = len(df) == expected_row_count
    checks["no_duplicated_ids"] = not df["ID"].duplicated().any()
    checks["no_missing_ids"] = not df["ID"].isnull().any()
    checks["no_missing_st"] = not df["ST-CODE"].isnull().any()
    checks["no_missing_rust"] = not df["RUST-CODE"].isnull().any()
    checks["no_empty_st"] = not (df["ST-CODE"].astype(str).str.strip() == "").any()
    checks["no_empty_rust"] = not (df["RUST-CODE"].astype(str).str.strip() == "").any()
    checks["no_markdown_fences_st"] = not df["ST-CODE"].astype(str).str.contains("```").any()
    checks["no_markdown_fences_rust"] = not df["RUST-CODE"].astype(str).str.contains("```").any()
    checks["st_resembles_structured_text"] = df["ST-CODE"].apply(lambda x: structural_score_st(x) > 0.3).all()
    checks["rust_resembles_rust"] = df["RUST-CODE"].apply(lambda x: structural_score_rust(x) > 0.3).all()

    for check_name, result in checks.items():
        print(f"{check_name}: {'PASS' if result else 'FAIL'}")

    all_passed = all(checks.values())
    print(f"\nOverall: {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")
    return checks

expected_rows = len(test_df)
validation_run1 = validate_submission(submission_run1, "submission_run1.csv", expected_rows)
validation_run2 = validate_submission(submission_run2, "submission_run2.csv", expected_rows)
validation_run3 = validate_submission(submission_run3, "submission_run3.csv", expected_rows)

VALIDATION REPORT: submission_run1.csv
correct_row_count: PASS
no_duplicated_ids: PASS
no_missing_ids: PASS
no_missing_st: PASS
no_missing_rust: PASS
no_empty_st: PASS
no_empty_rust: PASS
no_markdown_fences_st: PASS
no_markdown_fences_rust: PASS
st_resembles_structured_text: PASS
rust_resembles_rust: PASS

Overall: ALL CHECKS PASSED
VALIDATION REPORT: submission_run2.csv
correct_row_count: PASS
no_duplicated_ids: PASS
no_missing_ids: PASS
no_missing_st: PASS
no_missing_rust: PASS
no_empty_st: FAIL
no_empty_rust: PASS
no_markdown_fences_st: PASS
no_markdown_fences_rust: PASS
st_resembles_structured_text: FAIL
rust_resembles_rust: PASS

Overall: SOME CHECKS FAILED
VALIDATION REPORT: submission_run3.csv
correct_row_count: PASS
no_duplicated_ids: PASS
no_missing_ids: PASS
no_missing_st: PASS
no_missing_rust: PASS
no_empty_st: PASS
no_empty_rust: PASS
no_markdown_fences_st: PASS
no_markdown_fences_rust: PASS
st_resembles_structured_text: PASS
rust_resembles_rust: PASS

Overall: ALL CHECKS P

In [ ]:
TEAM_NAME = "TokenX"
SUBMISSION_ROOT = os.path.join(OUTPUT_DIR, "submission_packages")
os.makedirs(SUBMISSION_ROOT, exist_ok=True)

NOTEBOOK_SOURCE_PATH = None
for candidate in [
    r"C:\Users\VGST\Desktop\AI4Industrial Automation\AI4Industrial_Automation_Pipeline.ipynb",
]:
    if os.path.exists(candidate):
        NOTEBOOK_SOURCE_PATH = candidate
        break

def gather_environment_info():
    return {
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "peft_version": peft.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "total_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2) if torch.cuda.is_available() else None,
        "random_seed": GLOBAL_SEED
    }

def build_requirements_txt():
    return "transformers>=4.44.0\naccelerate>=0.33.0\npeft>=0.12.0\nbitsandbytes>=0.43.0\ndatasets>=2.20.0\nsentencepiece\nsentence-transformers\nscikit-learn\npandas\nnumpy\ntorch\ntqdm\nsafetensors\nopenpyxl\n"

def build_readme(run_number, run_label, model_used, csv_name, xlsx_name, env_info):
    return f"""# {TEAM_NAME} - Run {run_number}

## Shared Task
AI4Industrial Automation @ FIRE 2026 - Natural Language to IEC 61131-3 Structured Text (Task A) and Rust (Task B)

## Run Description
{run_label}

## System Information
- Python version: {env_info['python_version']}
- PyTorch version: {env_info['torch_version']}
- Transformers version: {env_info['transformers_version']}
- PEFT version: {env_info['peft_version']}
- CUDA available: {env_info['cuda_available']}
- CUDA version: {env_info['cuda_version']}
- GPU: {env_info['gpu_name']}
- VRAM: {env_info['total_vram_gb']} GB
- Random seed: {env_info['random_seed']}

## Model(s) Used
{model_used}

## Output Files
- {csv_name}
- {xlsx_name}

Both contain identical data. Columns: ID, ST-CODE, RUST-CODE

## Reproducibility Instructions
1. Create environment: conda create -n torch_env python=3.10
2. Activate: conda activate torch_env
3. Install dependencies: pip install -r requirements.txt
4. Place training/test CSV files at the paths configured in the notebook.
5. Run code/pipeline.ipynb cell by cell, top to bottom.
6. Output files are produced at outputs/ and included in this package.

## Restrictions Followed
- Only open-source models under 8B parameters were used.
- No proprietary or closed APIs were used for generation or training.
- No test labels or reference outputs were used during test inference.

## Notes
Local validation metrics are proxy metrics only, not the official FIRE 2026 leaderboard metrics.
"""

def package_run(run_number, run_label, model_used, submission_df, csv_name, xlsx_name):
    env_info = gather_environment_info()
    package_dir = os.path.join(SUBMISSION_ROOT, f"RUN-{TEAM_NAME}-{run_number}")
    code_dir = os.path.join(package_dir, "code")
    outputs_dir = os.path.join(package_dir, "outputs")

    if os.path.exists(package_dir):
        shutil.rmtree(package_dir)
    os.makedirs(code_dir, exist_ok=True)
    os.makedirs(outputs_dir, exist_ok=True)

    submission_df.to_csv(os.path.join(outputs_dir, csv_name), index=False)
    submission_df.to_excel(os.path.join(outputs_dir, xlsx_name), index=False, engine="openpyxl")

    if NOTEBOOK_SOURCE_PATH is not None:
        shutil.copy(NOTEBOOK_SOURCE_PATH, os.path.join(code_dir, "pipeline.ipynb"))
    else:
        with open(os.path.join(code_dir, "NOTEBOOK_MISSING.txt"), "w") as f:
            f.write("Notebook not found at configured path. Add your .ipynb manually before submitting.\n")

    with open(os.path.join(code_dir, "requirements.txt"), "w") as f:
        f.write(build_requirements_txt())

    with open(os.path.join(package_dir, "README.md"), "w", encoding="utf-8") as f:
        f.write(build_readme(run_number, run_label, model_used, csv_name, xlsx_name, env_info))

    with open(os.path.join(package_dir, "environment_info.json"), "w") as f:
        json.dump(env_info, f, indent=2)

    zip_path = os.path.join(SUBMISSION_ROOT, f"RUN-{TEAM_NAME}-{run_number}.zip")
    if os.path.exists(zip_path):
        os.remove(zip_path)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(package_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, package_dir)
                zf.write(file_path, arcname)

    print(f"Built: {zip_path}")
    return zip_path

zip1 = package_run(1, "Best individual fine-tuned model (LoRA on Qwen2.5-Coder-1.5B, deterministic decoding, best prompt per task).",
                    f"{baseline1_model} (LoRA fine-tuned, 4-bit quantization)", submission_run1,
                    "submission_run1.csv", "submission_run1.xlsx")

zip2 = package_run(2, "Alternative model/prompt strategy (Qwen2.5-Coder-7B, few-shot prompt, sampled decoding, 4-bit quantization).",
                    f"{baseline2_model} (4-bit quantization)", submission_run2,
                    "submission_run2.csv", "submission_run2.xlsx")

zip3 = package_run(3, "Best ensemble/candidate-ranking system (multi-model, multi-prompt candidate generation with scoring for final selection).",
                    f"{baseline1_model} + {baseline2_model} ensemble", submission_run3,
                    "submission_run3.csv", "submission_run3.xlsx")

print("\nAll packages built:")
print(zip1)
print(zip2)
print(zip3)

Built: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_packages\RUN-TokenX-1.zip
Built: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_packages\RUN-TokenX-2.zip
Built: C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_packages\RUN-TokenX-3.zip

All packages built:
C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_packages\RUN-TokenX-1.zip
C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_packages\RUN-TokenX-2.zip
C:/Users/VGST/Desktop/AI4Industrial Automation/outputs\submission_packages\RUN-TokenX-3.zip


: 